# PHARVO-beta: POS Manual Discount Test

**Objective:** Verify that applying a **15% manual discount** in the POS **TotalsCard** on medicine `Napa 500mg` dynamically recalculates the line discount amount and updates the payable grand total accurately.

### Test Parameters
- **Medicine:** `Napa 500mg`
- **Manual Discount:** `15%`

### Prerequisites
```bash
pip install selenium webdriver-manager
```
Ensure PHARVO frontend is running at `http://localhost:5173` and backend at `http://localhost:8000`.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Configuration & Test Data ---
BASE_URL = "http://localhost:5173"
USERNAME = "rafi"
PASSWORD = "password"  # Replace with actual password
MEDICINE_SEARCH = "Napa 500mg"
DISCOUNT_PERCENT = 15

# Step 1: Open browser and maximize window
driver = webdriver.Chrome()
driver.maximize_window()

# Set explicit wait helper (up to 10 seconds)
wait = WebDriverWait(driver, 10)

try:
    print(f"[INFO] Starting POS Discount Test ({DISCOUNT_PERCENT}% on '{MEDICINE_SEARCH}')...")

    # Step 2: Open login page and sign in
    driver.get(f"{BASE_URL}/")

    username_field = wait.until(
        EC.visibility_of_element_located((By.ID, "username"))
    )
    password_field = wait.until(
        EC.visibility_of_element_located((By.ID, "password"))
    )

    username_field.clear()
    username_field.send_keys(USERNAME)

    password_field.clear()
    password_field.send_keys(PASSWORD)

    sign_in_button = wait.until(
        EC.element_to_be_clickable((By.ID, "sign-in-btn"))
    )
    sign_in_button.click()

    # Step 3: Navigate to POS / Sales module via sidebar
    pos_nav = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//aside//button[contains(., 'POS / Sales')]")
        )
    )
    pos_nav.click()

    wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//header//h1[contains(text(), 'POS / Sales')]")
        )
    )
    print("[INFO] Navigated to POS / Sales terminal.")

    # Step 4: Search for target medicine and add to cart
    pos_search_input = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@aria-label='Search medicine or brand' and contains(@class, 'pos-input')]")
        )
    )
    pos_search_input.clear()
    pos_search_input.send_keys(MEDICINE_SEARCH)

    target_row = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, f"//table[contains(@class, 'pos-table')]//tbody//tr[contains(@class, 'pos-row-tr') and .//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'napa')]]")
        )
    )

    add_btn = target_row.find_element(By.XPATH, ".//button[contains(., 'Add')]")
    add_btn.click()
    print(f"[INFO] Added '{MEDICINE_SEARCH}' to cart.")

    # Step 5: Read original Subtotal before discount
    subtotal_elem = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//div[contains(@class, 'pos-card')]//span[contains(text(), 'Subtotal')]/following-sibling::span[contains(@class, 'tabular-nums')]")
        )
    )
    subtotal_val = float(''.join(c for c in subtotal_elem.text if c.isdigit() or c == '.'))
    print(f"[INFO] Initial Cart Subtotal: ?{subtotal_val:.2f}")

    # Step 6: Locate and fill the Manual Discount Percentage input
    discount_input = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@aria-label='Manual discount percent']")
        )
    )
    discount_input.clear()
    discount_input.send_keys(str(DISCOUNT_PERCENT))
    print(f"[INFO] Applied {DISCOUNT_PERCENT}% manual discount.")

    time.sleep(0.5)  # Allow calculation state to update

    # Step 7: Read updated discount breakdown & Grand Total
    manual_disc_elem = driver.find_element(
        By.XPATH, "//div[contains(@class, 'pos-card')]//span[contains(text(), 'Manual Discount')]/following-sibling::span[contains(@class, 'tabular-nums')]"
    )
    total_disc_elem = driver.find_element(
        By.XPATH, "//div[contains(@class, 'pos-card')]//span[contains(text(), 'Total Discount')]/following-sibling::span[contains(@class, 'tabular-nums')]"
    )
    grand_total_elem = driver.find_element(
        By.XPATH, "//div[contains(@class, 'pos-card')]//span[text()='Total']/following-sibling::span[contains(@class, 'tabular-nums')]"
    )

    discount_amount_val = float(''.join(c for c in manual_disc_elem.text if c.isdigit() or c == '.'))
    grand_total_val = float(''.join(c for c in grand_total_elem.text if c.isdigit() or c == '.'))

    # Step 8: Calculate expected values
    expected_discount = round(subtotal_val * (DISCOUNT_PERCENT / 100.0), 2)
    expected_grand_total = round(subtotal_val - expected_discount, 2)

    # Step 9: Verify calculations
    print(f"[INFO] Displayed Discount: {manual_disc_elem.text.strip()}")
    print(f"[INFO] Displayed Grand Total: {grand_total_elem.text.strip()}")
    print(f"[INFO] Expected Grand Total: ?{expected_grand_total:.2f}")

    # Tolerant float comparison
    if abs(grand_total_val - expected_grand_total) <= 0.1 and abs(discount_amount_val - expected_discount) <= 0.1:
        print(f"PASS: {DISCOUNT_PERCENT}% discount calculation verified successfully.")
        print(f"      - Original Subtotal: ?{subtotal_val:.2f}")
        print(f"      - Discount Applied: -?{discount_amount_val:.2f}")
        print(f"      - Final Payable Total: ?{grand_total_val:.2f}")
    else:
        print(f"FAIL: Calculation discrepancy (Expected Total ?{expected_grand_total}, Got ?{grand_total_val}).")

except Exception as error:
    print(f"FAIL: Discount test encountered error: {error}")

finally:
    # Step 10: Close browser
    print("[INFO] Cleaning up and closing browser...")
    driver.quit()
